# vfairness · Module 10 · Test objects and access depth

How much a fairness test can say depends on what you can see of the system. This notebook
walks one real source (the UCI Adult census income task, scored by a published logistic
regression) up the access tiers of the *Precision Is Not Proof* framework, and then runs the
Pulse triage on synthetic files whose planted bias is known in advance.

| Tier | You hold | What it lets you claim |
|:-|:-|:-|
| A0 Attested | a document saying the system is fair | nothing measured |
| A1 Behavioural | decisions (plus the group of each case; labels if you have them) | selection-rate and error-rate gaps |
| A2 Scored | plus probabilities or scores | calibration by group, gaps inside score bands |
| A3 Internal | plus weights or activations | attribution, exact counterfactuals |
| A4 Provenance | plus the training data | where a learned association came from |

Two more axes travel with every claim: the **evidence grade** (E1 observational, E2 controlled,
for example counterfactual pairs) and the **validity grade** (V0 unpinned, V1 pinned model
version). The headline claim is the weakest of the three. A null result has to state what it
could have detected: "not seen" is not "seen and fine".

1. **Loading the test objects**
2. **A1, decisions and group only**: selection-rate disparity with a bootstrap interval
3. **A1 with labels**: error-rate parity
4. **A2 with probabilities**: calibration by group
5. **A3 with the weights**: rebuild the model, attribute the gap, flip a proxy feature
6. **Synthetic files with a known answer**: Pulse on a control and on planted biases
7. **A declared ceiling**: a file with no race column
8. **Optional, needs network**: COMPAS from its origin
9. **What each tier was able to claim**


## 1. Loading the test objects

The files live in `docs/site/test-objects/data/` in the repository and are published at
`https://vfairness.validant.ai/test-objects/data/`. The helper reads the local folder when it
exists (from the repository root or from `notebooks/`) and otherwise fetches the published copy.


In [2]:
import os, io, json, time, logging, warnings, contextlib, hashlib, urllib.request
warnings.filterwarnings("ignore")          # library warnings name local file paths
logging.disable(logging.WARNING)
import numpy as np
import pandas as pd

DATA_URL = "https://vfairness.validant.ai/test-objects/data/"
_CANDIDATES = ["docs/site/test-objects/data", "../docs/site/test-objects/data"]
DATA_DIR = next((p for p in _CANDIDATES if os.path.isdir(p)), None)

def data_path(rel):
    """Local copy when the repository is checked out, published URL otherwise."""
    return os.path.join(DATA_DIR, rel) if DATA_DIR else DATA_URL + rel

def load_csv(rel, **kw):
    return pd.read_csv(data_path(rel), **kw)

def load_json(rel):
    if DATA_DIR:
        with open(data_path(rel)) as fh:
            return json.load(fh)
    with urllib.request.urlopen(data_path(rel), timeout=60) as r:
        return json.load(r)

print("data source:", DATA_DIR if DATA_DIR else DATA_URL)
manifest = load_json("manifest.json")
print("files in manifest:", len(manifest["files"]))
for f in manifest["files"]:
    if f["path"].startswith(("predictive/adult", "synthetic/", "scripts/get_compas")):
        print(f"  {f['tier']:3s} {f['path']}")


data source: ../docs/site/test-objects/data
files in manifest: 25
  A2  predictive/adult_test_with_predictions.csv
  A3  predictive/adult_logreg_model.json
  A2  scripts/get_compas.py
  A1  synthetic/clean.csv
  A1  synthetic/gender_penalty.csv
  A1  synthetic/race_penalty.csv
  A1  synthetic/age_cliff.csv
  A1  synthetic/disability_penalty.csv
  A1  synthetic/zip_redlining.csv
  A1  synthetic/surname_proxy.csv
  A1  synthetic/photo_laundering.csv


In [3]:
import vfairness
from vfairness import FairnessAnalyzer, CalibrationAnalyzer
print("vfairness", vfairness.__version__)

adult = load_csv("predictive/adult_test_with_predictions.csv")
print("rows:", len(adult))
print("columns:", list(adult.columns))
print("rows by sex :", adult["sex"].value_counts().to_dict())
print("rows by race:", adult["race"].value_counts().to_dict())


vfairness 0.1.0
rows: 15060
columns: ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'y_true', 'y_prob', 'y_pred']
rows by sex : {'Male': 10147, 'Female': 4913}
rows by race: {'White': 12970, 'Black': 1411, 'Asian-Pac-Islander': 408, 'Amer-Indian-Eskimo': 149, 'Other': 122}


## 2. A1: decisions and group only

Pretend we hold only `y_pred` and the group of each person, as a regulator receiving a decision
log would. The only measurable quantity is the selection rate per group. The interval is a
percentile bootstrap (1,000 resamples within each group, fixed seed), so it is reproducible.

We also print the **limiting magnitude**: the smallest gap this sample could have detected with
80 percent power at the 5 percent level. A gap smaller than that is not evidence of fairness; it
is outside what the test can see.


In [4]:
Z_ALPHA, Z_POWER = 1.959964, 0.841621      # two-sided 5 percent, 80 percent power

def bootstrap_gap(values, groups, a, b, n_boot=1000, seed=0):
    """rate(a) minus rate(b), with a percentile bootstrap interval."""
    rng = np.random.default_rng(seed)
    va = np.asarray(values[groups == a], dtype=float)
    vb = np.asarray(values[groups == b], dtype=float)
    boots = np.array([rng.choice(va, va.size).mean() - rng.choice(vb, vb.size).mean()
                      for _ in range(n_boot)])
    return va.mean() - vb.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)

def limiting_gap(p, n1, n2):
    """Smallest difference in proportions detectable at 80 percent power."""
    return (Z_ALPHA + Z_POWER) * np.sqrt(p * (1 - p) * (1 / n1 + 1 / n2))

def selection_table(frame, attr, ref):
    rows = []
    rates = frame.groupby(attr)["y_pred"].agg(["mean", "size"])
    for g, r in rates.iterrows():
        if g == ref:
            continue
        gap, lo, hi = bootstrap_gap(frame["y_pred"].values, frame[attr].values, ref, g)
        rows.append({"group": g, "n": int(r["size"]), "selection_rate": round(r["mean"], 4),
                     "gap_vs_" + str(ref): round(gap, 4), "ci_low": round(lo, 4), "ci_high": round(hi, 4),
                     "ratio": round(r["mean"] / rates.loc[ref, "mean"], 3),
                     "limiting_gap": round(limiting_gap(frame["y_pred"].mean(), r["size"], rates.loc[ref, "size"]), 4)})
    print(f"reference {ref}: n={int(rates.loc[ref, 'size'])}, selection rate {rates.loc[ref, 'mean']:.4f}")
    return pd.DataFrame(rows)

print(selection_table(adult, "sex", "Male").to_string(index=False))
print()
print(selection_table(adult, "race", "White").to_string(index=False))


reference Male: n=10147, selection rate 0.2599
 group    n  selection_rate  gap_vs_Male  ci_low  ci_high  ratio  limiting_gap
Female 4913          0.0812       0.1787  0.1669   0.1897  0.313        0.0195



reference White: n=12970, selection rate 0.2130
             group    n  selection_rate  gap_vs_White  ci_low  ci_high  ratio  limiting_gap
Amer-Indian-Eskimo  149          0.0872        0.1257  0.0772   0.1711  0.410        0.0926
Asian-Pac-Islander  408          0.2745       -0.0616 -0.1066  -0.0173  1.289        0.0565
             Black 1411          0.0964        0.1166  0.0988   0.1348  0.453        0.0315
             Other  122          0.1066        0.1064  0.0508   0.1573  0.500        0.1022


Read the race table with the `limiting_gap` column in view. A group whose interval spans zero
and whose limiting gap is large has not been shown to be treated equally; the test simply could
not resolve a difference of that size. At A1 without labels we also cannot say whether a gap is
*justified* by different base rates: that needs the outcome, which is the next tier.


## 3. A1 with labels: error-rate parity

With the true outcome `y_true` we can ask whether the model makes the same kinds of mistakes for
each group (equalised odds, equal opportunity). `FairnessAnalyzer` computes these natively with
bootstrap intervals and returns a verdict that can be `insufficient_evidence` when the interval
straddles the threshold.


In [5]:
fa = FairnessAnalyzer(adult["y_true"].values, adult["y_pred"].values, adult["sex"].values,
                      y_prob=adult["y_prob"].values)
res = fa.compute_all_metrics(include_ci=True)
for name in ["demographic_parity_difference", "equalized_odds_difference", "equal_opportunity_difference"]:
    r = res[name]
    lo, hi = r.confidence_interval
    print(f"{name:32s} {r.value:.4f}  95% CI [{lo:.4f}, {hi:.4f}]  threshold {r.threshold}  verdict {r.verdict}")

# The A1 number from section 2 and the analyzer's number must agree.
own_gap = adult.groupby("sex")["y_pred"].mean()
print("\nsection 2 sex gap:", round(own_gap["Male"] - own_gap["Female"], 6),
      " analyzer DP:", round(res["demographic_parity_difference"].value, 6))

print("\nper-group error rates")
for g, sub in adult.groupby("sex"):
    tpr = sub.loc[sub.y_true == 1, "y_pred"].mean()
    fpr = sub.loc[sub.y_true == 0, "y_pred"].mean()
    print(f"  {g:6s} base rate {sub.y_true.mean():.4f}  TPR {tpr:.4f}  FPR {fpr:.4f}")


demographic_parity_difference    0.1787  95% CI [0.1671, 0.1903]  threshold 0.1  verdict unfair
equalized_odds_difference        0.0795  95% CI [0.0390, 0.1199]  threshold 0.1  verdict insufficient_evidence
equal_opportunity_difference     0.0752  95% CI [0.0300, 0.1218]  threshold 0.1  verdict insufficient_evidence

section 2 sex gap: 0.178667  analyzer DP: 0.178667

per-group error rates
  Female base rate 0.1134  TPR 0.5350  FPR 0.0232
  Male   base rate 0.3097  TPR 0.6102  FPR 0.1027


The selection-rate gap is partly a base-rate gap: the label itself differs by sex in this
census sample. Error-rate parity asks a narrower question, and its verdict here can be weaker
than the demographic parity verdict even though it uses more information. More access does not
mean a louder finding; it means a more specific one.


## 4. A2 with probabilities: calibration by group

With `y_prob` we can check whether a score of 0.3 means the same thing for a woman as for a man.
`CalibrationAnalyzer` reports expected calibration error (ECE) per group. We also print
calibration in the large (mean score against observed rate) and the observed rate inside each
score band, which is where a calibration gap would show.


In [6]:
ca = CalibrationAnalyzer(y_true=adult["y_true"].values, y_prob=adult["y_prob"].values,
                         protected_attr=adult["sex"].values, attribute_name="sex")
disp = ca.analyze_disparity()
print("ECE by group:", {k: round(float(v), 4) for k, v in disp.group_ece.items()})
print("ECE disparity:", round(disp.ece_disparity, 4), " most miscalibrated:", disp.most_miscalibrated_group)
print("library recommendation:", disp.recommendations)

print("\ncalibration in the large")
print(adult.groupby("sex").agg(mean_score=("y_prob", "mean"), observed_rate=("y_true", "mean"), n=("y_true", "size")).round(4).to_string())

bands = pd.cut(adult["y_prob"], [0, 0.1, 0.3, 0.5, 0.7, 1.0], include_lowest=True)
tab = adult.groupby([bands, "sex"], observed=True)["y_true"].agg(["mean", "size"]).unstack("sex")
print("\nobserved positive rate inside each score band")
print(tab.round(3).to_string())


ECE by group: {'Female': 0.0204, 'Male': 0.0115}
ECE disparity: 0.0089  most miscalibrated: Female
library recommendation: ['LOW: Calibration disparity is acceptable (0.009).']

calibration in the large
        mean_score  observed_rate      n
sex                                     
Female      0.1279         0.1134   4913
Male        0.3057         0.3097  10147

observed positive rate inside each score band
                mean          size      
sex           Female   Male Female  Male
y_prob                                  
(-0.001, 0.1]  0.017  0.028   3446  3700
(0.1, 0.3]     0.137  0.221    802  2327
(0.3, 0.5]     0.338  0.409    266  1483
(0.5, 0.7]     0.606  0.583    188  1183
(0.7, 1.0]     0.872  0.845    211  1454


## 5. A3: the weights

`adult_logreg_model.json` publishes every weight. The model was never given `sex` or `race`.
First we rebuild `predict()` from the JSON and require that it reproduces the published
`y_prob`; if it does not, nothing that follows is about this model.


In [7]:
model = load_json("predictive/adult_logreg_model.json")
print("model:", model["model"])
print("not used:", model["not_used"])
print("how to predict:", model["how_to_predict"])

def contributions(frame):
    """Per-feature contribution to the logit (coefficient times encoded feature)."""
    cols = {}
    for f in model["features_numeric"]:
        x = (frame[f].values - model["scaler_mean"][f]) / model["scaler_scale"][f]
        cols["num__" + f] = model["coefficients"]["num__" + f] * x
    for f in model["features_categorical"]:
        for cat in model["categories"][f]:
            key = f"cat__{f}_{cat}"
            cols[key] = model["coefficients"][key] * (frame[f].values == cat)
    return pd.DataFrame(cols, index=frame.index)

def predict_proba(frame):
    z = model["intercept"] + contributions(frame).sum(axis=1)
    return 1 / (1 + np.exp(-z))

rebuilt = predict_proba(adult)
max_err = float(np.abs(rebuilt - adult["y_prob"]).max())
mismatched = int(((rebuilt >= model["threshold"]).astype(int) != adult["y_pred"]).sum())
print(f"\nmax |rebuilt - published y_prob| = {max_err:.2e}   decisions that differ: {mismatched}")
assert max_err < 1e-5 and mismatched == 0, "the JSON does not reproduce the published scores"
print("reproduction check: PASSED (published y_prob is rounded to 6 decimals)")


model: logistic regression, sklearn 1.8.0
not used: ['sex', 'race', 'native_country', 'fnlwgt', 'education']
how to predict: z = intercept + sum(coef * feature), numeric features standardised with scaler_mean / scaler_scale, categorical one-hot; y_prob = 1 / (1 + exp(-z)); y_pred = y_prob >= threshold

max |rebuilt - published y_prob| = 5.00e-07   decisions that differ: 0
reproduction check: PASSED (published y_prob is rounded to 6 decimals)


### 5a. Attribution: which features carry the sex gap

For a linear model the gap in mean logit between two groups splits exactly into per-feature
terms: mean contribution for men minus mean contribution for women. This is an internal check
only white-box access allows, and it is exact, not an approximation.


In [8]:
C = contributions(adult)
by_sex = C.groupby(adult["sex"]).mean()
term = (by_sex.loc["Male"] - by_sex.loc["Female"])

def source(k):
    """The original feature a coefficient belongs to (one-hot terms grouped back)."""
    if k.startswith("num__"):
        return k[5:]
    rest = k[5:]
    return next(f for f in model["features_categorical"] if rest.startswith(f + "_"))
block = term.groupby(source).sum().sort_values(ascending=False)
total = term.sum()
logit = model["intercept"] + C.sum(axis=1)
print("mean logit gap (Male minus Female):", round(float(total), 4),
      " check:", round(float(logit[adult.sex == "Male"].mean() - logit[adult.sex == "Female"].mean()), 4))
print("\nshare of the gap by source feature")
print(pd.DataFrame({"logit_gap": block.round(4), "share": (block / total).round(3)}).to_string())
print("\nlargest single terms")
print(term.reindex(term.abs().sort_values(ascending=False).index).head(6).round(4).to_string())


mean logit gap (Male minus Female): 1.6897  check: 1.6897

share of the gap by source feature
                logit_gap  share
marital_status     0.9969  0.590
capital_gain       0.2323  0.137
hours_per_week     0.1819  0.108
relationship       0.1260  0.075
occupation         0.0762  0.045
age                0.0614  0.036
capital_loss       0.0252  0.015
education_num     -0.0032 -0.002
workclass         -0.0070 -0.004

largest single terms
cat__marital_status_Married-civ-spouse    0.5834
num__capital_gain                         0.2323
cat__marital_status_Never-married         0.2126
num__hours_per_week                       0.1819
cat__marital_status_Divorced              0.1261
cat__occupation_Other-service             0.0898


### 5b. Counterfactual flip of a proxy: Husband and Wife

`relationship` takes the value `Husband` or `Wife`, so it encodes sex for every married person
even though `sex` is not a feature. With the weights we can run a controlled experiment (evidence
grade E2): swap `Husband` and `Wife` for every row, keep everything else fixed, and see what
happens to the selection-rate gap.


In [9]:
print("relationship coefficients:",
      {k.split("_")[-1]: round(v, 4) for k, v in model["coefficients"].items() if "relationship" in k})
print("married rows: Husband", int((adult.relationship == "Husband").sum()), " Wife", int((adult.relationship == "Wife").sum()))

flipped = adult.copy()
flipped["relationship"] = adult["relationship"].replace({"Husband": "Wife", "Wife": "Husband"})
neutral = adult.copy()
neutral["relationship"] = adult["relationship"].replace({"Wife": "Husband"})   # one shared spouse code

def sex_gap(frame):
    yp = (predict_proba(frame) >= model["threshold"]).astype(int)
    r = yp.groupby(adult["sex"]).mean()
    return r["Male"], r["Female"], r["Male"] - r["Female"]

rows = []
for name, frame in [("as deployed", adult), ("Husband and Wife swapped", flipped), ("one spouse code (Wife set to Husband)", neutral)]:
    m_, f_, g_ = sex_gap(frame)
    rows.append({"scenario": name, "male_rate": round(m_, 4), "female_rate": round(f_, 4), "gap": round(g_, 4)})
flip = pd.DataFrame(rows)
print(flip.to_string(index=False))
base = flip.loc[0, "gap"]
for i in (1, 2):
    print(f"change in gap, {flip.loc[i, 'scenario']}: {flip.loc[i, 'gap'] - base:+.4f} ({(flip.loc[i, 'gap'] - base) / base:+.1%} of the deployed gap)")


relationship coefficients: {'Husband': 0.0988, 'Not-in-family': 0.0749, 'Other-relative': -0.7715, 'Own-child': -1.0091, 'Unmarried': -0.2974, 'Wife': 0.5752}
married rows: Husband 6203  Wife 685
                             scenario  male_rate  female_rate    gap
                          as deployed     0.2599       0.0812 0.1787
             Husband and Wife swapped     0.3338       0.0625 0.2713
one spouse code (Wife set to Husband)     0.2599       0.0625 0.1974
change in gap, Husband and Wife swapped: +0.0926 (+51.8% of the deployed gap)
change in gap, one spouse code (Wife set to Husband): +0.0187 (+10.5% of the deployed gap)


Read the sign. The naive expectation is that the Husband code is what pushes men up. The weights
say otherwise: the `Wife` coefficient is the larger one, so the Husband/Wife distinction on its
own works in women's favour and removing it widens the gap. The gap travels through the
features that differ in *distribution* between the sexes (the attribution table above), not
through the one that names sex most directly. A black-box test (A1) could not have separated
those two stories; it would only have seen the total.


## 6. Synthetic files with a known answer

The `synthetic/` files are one hiring screen generated eight times: a clean control and seven
copies with one planted mechanism each. `_bias_flags` is the answer key and
`true_qualification_score` is the latent truth; both are removed before analysis, exactly as a
real audit would never see them.

We run the Pulse triage (`run_pulse`) on four of them. Pulse is slow (it runs the whole battery),
so we pass only the two protected attributes the planted mechanisms below act on (gender and
ZIP) and drop the unique identifier columns, which carry no decision information and are
trivially associated with every attribute. Every file is read with the same settings, so the
control is comparable with the planted files.


In [10]:
from vfairness.operations.pulse import run_pulse

ANSWER_KEY = ["true_qualification_score", "_bias_flags"]
IDENTIFIERS = ["applicant_id", "first_name", "application_date"]
ATTRS = ["gender", "zip_minority_majority"]

def pulse_file(name, attrs=ATTRS):
    raw = load_csv(f"synthetic/{name}.csv")
    frame = raw.drop(columns=ANSWER_KEY + IDENTIFIERS, errors="ignore")
    t0 = time.time()
    out = run_pulse(frame, {"protected_attributes": attrs, "domain": "employment"})["data"]
    rows = []
    for v in out.get("perVariable", []):
        rows.append({"attribute": v["attribute"], "reference": v.get("referenceGroup"),
                     "worst": v.get("worstGroup"), "gap": round(v["gap"], 4) if v.get("gap") is not None else None,
                     "ci_low": round(v["ciLow"], 4) if v.get("ciLow") is not None else None,
                     "ci_high": round(v["ciHigh"], 4) if v.get("ciHigh") is not None else None,
                     "p_adj": round(v["pValueAdjusted"], 4) if v.get("pValueAdjusted") is not None else None,
                     "significant": v.get("significant"), "tone": v.get("tone")})
    print(f"=== {name}  ({len(frame)} rows, {time.time() - t0:.0f} s)")
    print("planted (manifest):", next((f["planted"] for f in manifest["files"] if f["path"] == f"synthetic/{name}.csv"), "not listed"))
    print("Pulse headline    :", out["verdict"]["headline"])
    print(pd.DataFrame(rows).to_string(index=False))
    assessed = [v["attribute"] for v in out.get("perVariable", [])]
    print("requested:", attrs, " assessed:", assessed,
          " not assessable:", [a for a in attrs if a not in assessed])
    return raw, out

pulse = {}
for name in ["clean", "gender_penalty", "zip_redlining"]:
    pulse[name] = pulse_file(name)
    print()


=== clean  (500 rows, 35 s)
planted (manifest): None. A correct detector reports nothing here.
Pulse headline    : Within fairness budget on the assessed attributes.
            attribute reference    worst    gap  ci_low  ci_high  p_adj  significant tone
               gender    Female     Male 0.0045 -0.0719   0.0838    1.0        False pass
zip_minority_majority  majority minority 0.0578 -0.0235   0.1417    1.0        False pass
requested: ['gender', 'zip_minority_majority']  assessed: ['gender', 'zip_minority_majority']  not assessable: []



=== gender_penalty  (500 rows, 55 s)
planted (manifest): Half of female invites in tech roles removed (B1).
Pulse headline    : Within fairness budget on the assessed attributes.
            attribute reference    worst    gap  ci_low  ci_high  p_adj  significant tone
               gender      Male   Female 0.0387 -0.0437   0.1092    1.0        False pass
zip_minority_majority  minority majority 0.0357 -0.0454   0.1227    1.0        False pass
requested: ['gender', 'zip_minority_majority']  assessed: ['gender', 'zip_minority_majority']  not assessable: []



=== zip_redlining  (500 rows, 42 s)
planted (manifest): Minority-majority ZIP invites halved; race itself untouched (B4).
Pulse headline    : Not fit to deploy as-is: material disparity for "minority".
            attribute reference    worst    gap  ci_low  ci_high  p_adj  significant     tone
               gender      Male   Female 0.0163 -0.0619   0.0916  1.000        False     pass
zip_minority_majority  majority minority 0.1903  0.1183   0.2615  0.004         True critical
requested: ['gender', 'zip_minority_majority']  assessed: ['gender', 'zip_minority_majority']  not assessable: []



Compare the three runs on the planted attribute.

* **zip_redlining**: the ZIP gap is large and survives multiple-comparison correction. The
  planted mechanism is found.
* **clean**: nothing is significant on either attribute. This is what a correct detector must
  report on the control.
* **gender_penalty**: Pulse does **not** confirm a gender gap. The next cell shows why, and what
  it would have taken to see it.


In [11]:
raw_gp, _ = pulse["gender_penalty"]
raw_cl, _ = pulse["clean"]
yes = lambda s: (s == "Yes").astype(int)

def gender_gap_by_stratum(raw, label):
    tech = raw["role_applied"].str.contains("Engineer|Data Scientist")
    out = []
    for stratum, mask in [("all roles", tech | ~tech), ("technical roles", tech), ("other roles", ~tech)]:
        sub = raw[mask & raw.gender.isin(["Male", "Female"])]
        dec = yes(sub["invite_decision"]).values
        gap, lo, hi = bootstrap_gap(dec, sub["gender"].values, "Male", "Female")
        nm, nf = int((sub.gender == "Male").sum()), int((sub.gender == "Female").sum())
        out.append({"file": label, "stratum": stratum, "n_male": nm, "n_female": nf,
                    "gap_male_minus_female": round(gap, 4), "ci_low": round(lo, 4), "ci_high": round(hi, 4),
                    "limiting_gap": round(limiting_gap(dec.mean(), nm, nf), 4)})
    return out

strata = pd.DataFrame(gender_gap_by_stratum(raw_cl, "clean") + gender_gap_by_stratum(raw_gp, "gender_penalty"))
print(strata.to_string(index=False))

flags = raw_gp["_bias_flags"].fillna("")
print("\nanswer key, read only now: rows flagged B1:", int((flags == "B1").sum()),
      "| by gender:", raw_gp.loc[flags == "B1", "gender"].value_counts().to_dict(),
      "| invited among them:", int(yes(raw_gp.loc[flags == "B1", "invite_decision"]).sum()))


          file         stratum  n_male  n_female  gap_male_minus_female  ci_low  ci_high  limiting_gap
         clean       all roles     232       255                -0.0045 -0.0806   0.0782        0.1165
         clean technical roles     109       123                 0.1118  0.0027   0.2238        0.1655
         clean     other roles     123       132                -0.1110 -0.2233   0.0087        0.1635
gender_penalty       all roles     239       244                 0.0387 -0.0437   0.1092        0.1131
gender_penalty technical roles     117       125                 0.1466  0.0462   0.2567        0.1480
gender_penalty     other roles     122       119                -0.0745 -0.1913   0.0419        0.1689

answer key, read only now: rows flagged B1: 125 | by gender: {'Female': 125} | invited among them: 18


Read the two files side by side. The penalty is confined to technical roles, so across all roles
it is diluted below the all-roles `limiting_gap`. Stratifying by a legitimate covariate (the role
applied for) points at technical roles, but the technical-roles gap in the **clean control** is
of a similar size and its interval also excludes zero: with about 120 people per gender in that
cell, the planted effect sits near the limiting magnitude, and chance alone produces gaps of that
size. Neither Pulse nor the stratified check can confirm the planted mechanism in 500 rows.
The correct reading is "not seen at this sample size, and only gaps above the limiting gap could
have been", never "no gender bias". The answer key confirms the mechanism is there.


## 7. A declared ceiling: no race column

In `surname_proxy.csv` the decision penalises surnames typical of Black and Hispanic applicants,
and the `race_ethnicity` column has been removed. We ask Pulse for the same two attributes plus `race_ethnicity`.


In [12]:
raw_sp, out_sp = pulse_file("surname_proxy", ATTRS + ["race_ethnicity"])
print("\nrace_ethnicity in the file:", "race_ethnicity" in raw_sp.columns)
print("Pulse coverage notes:")
for line in out_sp.get("scope", {}).get("notCovered", []):
    print("  -", line)


=== surname_proxy  (500 rows, 72 s)
planted (manifest): Surnames typical of Black and Hispanic applicants lose half their invites; race_ethnicity is not in the file (B5).
Pulse headline    : Within fairness budget on the assessed attributes.
            attribute reference    worst    gap  ci_low  ci_high  p_adj  significant tone
               gender    Female     Male 0.0392 -0.0376   0.1174    1.0        False pass
zip_minority_majority  minority majority 0.0385 -0.0369   0.1234    1.0        False pass
requested: ['gender', 'zip_minority_majority', 'race_ethnicity']  assessed: ['gender', 'zip_minority_majority']  not assessable: ['race_ethnicity']

race_ethnicity in the file: False
Pulse coverage notes:
  - Causal identification: the causal view is a structural sketch, correlational only
  - Regression / multiclass / ranking outputs (binary decisions only in this release)
  - Deployment, human-factors and systemic bias (NIST SP 1270): reachable only through the full assessment ques

In [13]:
# What CAN be said without a race column: decisions differ by surname.
top = raw_sp.groupby("last_name")["invite_decision"].agg(rate=lambda s: (s == "Yes").mean(), n="size")
top = top[top.n >= 15].sort_values("rate")
print(f"surnames with at least 15 applicants: {len(top)}")
print("lowest and highest invite rates by surname:")
print(pd.concat([top.head(4), top.tail(4)]).round(3).to_string())
print("\nCLAIM CEILING: race_ethnicity is absent, so a race finding is 'could not check'.")
print("A surname pattern is evidence about surnames. Turning it into a race claim needs either the")
print("race column (A1 with groups) or a validated surname-to-race imputation, which is itself a")
print("measurement with its own error and must be declared as such.")


surnames with at least 15 applicants: 18
lowest and highest invite rates by surname:
            rate   n
last_name           
Okafor     0.077  26
Rodriguez  0.094  32
Diallo     0.111  36
Garcia     0.114  44
Nguyen     0.344  32
Lee        0.357  28
Khan       0.375  16
Chen       0.625  24

CLAIM CEILING: race_ethnicity is absent, so a race finding is 'could not check'.
A surname pattern is evidence about surnames. Turning it into a race claim needs either the
race column (A1 with groups) or a validated surname-to-race imputation, which is itself a
measurement with its own error and must be declared as such.


Pulse reports the assessed attributes as within budget, and that statement is correct about
what it covers: gender and ZIP. It is silent on race because race is not in the file. The
planted mechanism is real and invisible at this access level. This is why the headline claim
has to carry its scope: "within budget on gender and ZIP; race could not be checked".


## 8. Optional, needs network: COMPAS from its origin

`scripts/get_compas.py` downloads ProPublica's COMPAS file, checks its hash, applies ProPublica's
filters and adds `y_true`, `score` and `y_pred`. Nothing is re-hosted, because the origin file
names real people. `score` is a decile rank, not a probability, so this source is A1 with an
ordinal score: no calibration claim is possible. If the download fails the cell says so and the
notebook continues.


In [14]:
import tempfile, types
compas = None
try:
    if DATA_DIR:
        with open(data_path("scripts/get_compas.py")) as fh:
            src = fh.read()
    else:
        src = urllib.request.urlopen(data_path("scripts/get_compas.py"), timeout=30).read().decode()
    mod = types.ModuleType("get_compas")
    exec(compile(src, "get_compas.py", "exec"), mod.__dict__)
    with tempfile.TemporaryDirectory() as tmp:
        target = os.path.join(tmp, "compas_with_predictions.csv")
        with contextlib.redirect_stdout(io.StringIO()):
            mod.main(target)
        compas = pd.read_csv(target)
except BaseException as exc:   # SystemExit from a hash mismatch included
    print(f"could not check: COMPAS download failed ({type(exc).__name__})")

if compas is not None:
    print("rows after ProPublica's filters:", len(compas))
    sub = compas[compas.race.isin(["African-American", "Caucasian"])]
    for race, g in sub.groupby("race"):
        fpr = g.loc[g.y_true == 0, "y_pred"].mean()
        fnr = 1 - g.loc[g.y_true == 1, "y_pred"].mean()
        print(f"  {race:17s} n={len(g):5d}  base rate {g.y_true.mean():.3f}  FPR {fpr:.3f}  FNR {fnr:.3f}")
    print("calibration: could not check (score is a decile rank, not a probability)")


rows after ProPublica's filters: 6172
  African-American  n= 3175  base rate 0.523  FPR 0.423  FNR 0.285
  Caucasian         n= 2103  base rate 0.391  FPR 0.220  FNR 0.496
calibration: could not check (score is a decile rank, not a probability)


## 9. What each tier was able to claim

The table below is built from the numbers computed above, not typed in.


In [15]:
summary = pd.DataFrame([
    {"tier": "A1 decisions + group", "evidence": "E1", "claim": f"selection-rate gap (Male minus Female) {own_gap['Male'] - own_gap['Female']:.4f}"},
    {"tier": "A1 + labels", "evidence": "E1", "claim": f"equalised odds difference {res['equalized_odds_difference'].value:.4f}, verdict {res['equalized_odds_difference'].verdict}"},
    {"tier": "A2 + probabilities", "evidence": "E1", "claim": f"ECE Female {float(disp.group_ece['Female']):.4f} vs Male {float(disp.group_ece['Male']):.4f}"},
    {"tier": "A3 + weights", "evidence": "E2", "claim": f"swapping Husband/Wife moves the gap by {flip.loc[1, 'gap'] - flip.loc[0, 'gap']:+.4f}"},
])
print(summary.to_string(index=False))
print("\nvalidity: V1 for the Adult model (the published JSON reproduces every published score, checked in section 5)")
print("A4 (training data) would let us ask where the household features learned their weights; not run here.")


                tier evidence                                                           claim
A1 decisions + group       E1                   selection-rate gap (Male minus Female) 0.1787
         A1 + labels       E1 equalised odds difference 0.0795, verdict insufficient_evidence
  A2 + probabilities       E1                                ECE Female 0.0204 vs Male 0.0115
        A3 + weights       E2                  swapping Husband/Wife moves the gap by +0.0926

validity: V1 for the Adult model (the published JSON reproduces every published score, checked in section 5)
A4 (training data) would let us ask where the household features learned their weights; not run here.
